# 09 · Error taxonomy: what causes the false alarms and the missed matches

> **Status (2026-11-25):** the conclusions and findings in this notebook were written on the **uncleaned** pairs (`pairs_scoped.parquet`). The code now runs on the cleaned pairs (`config.PAIRS_INPUT` = `pairs_clean.parquet`); after the re-run, the numbers in the conclusions must be updated.

The second deliverable next to the cost curve: **which kinds of pairs** cause the false alarms at near-perfect recall, which true matches the model still misses, and how much of it no feature could ever fix.

- **Input:** the validation scores saved by notebook 07, section 11 (`data/processed/scores_dev.parquet`): LightGBM, largest feature set, split by person cluster, 5 repetitions. No model is trained here, except once for the SHAP analysis (section 8).
- **Data:** whatever notebook 07 ran on: the cleaned pairs, Vova's train + val only, split by person group. Re-run 07 first whenever the data or features change.
- **Never on the test set** until the model is frozen; then once, at the end.

Abbreviations: **FA** = false alarm (negative pair with an alarm); **miss** = positive pair without an alarm; **FA rate** = false alarms / negative pairs; **pp** = percentage points; **SHAP** = contribution of each feature to one prediction.

In [ ]:
import sys, re, time
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from screening import config as C
from screening.lists import values

pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)
RECALLS, TARGET = [0.95, 0.98, 0.99, 0.995], 0.99

scores = pd.read_parquet(C.DATA_PROCESSED / "scores_dev.parquet")
FEATURE_SET = scores["feature_set"].iloc[0]
props = ["name", "alias", "birthDate", "country", "nationality", "idNumber"]
cols = ["pair_id", "left_schema"] + [f"{s}_{p}" for s in ("left", "right") for p in props]
pairs = pd.read_parquet(C.PAIRS_INPUT, columns=cols)
feats = pd.read_parquet(C.FEATURES_NAME)
if FEATURE_SET == "name+record":
    feats = feats.merge(pd.read_parquet(C.FEATURES_RECORD), on="pair_id")
info = pairs.merge(feats, on="pair_id")
info["person"] = info["left_schema"] == "Person"
df = scores.merge(info, on="pair_id", how="left", validate="many_to_one")
assert df["n_sim_sort"].notna().mean() > 0.95, "scores and features do not match: re-run notebook 07"
print(f"{len(scores):,} validation scores from {scores['rep'].nunique()} splits; model {scores['model'].iloc[0]}, "
      f"features '{FEATURE_SET}', input {C.PAIRS_INPUT.name}")

## 1 · Record fields of each pair

For the taxonomy, every pair gets a simple status for its record fields, computed from the raw values (independent of the feature set): **same** (a value occurs on both sides; birth dates also if one is a prefix of the other, e.g. 1968 vs 1968-03-02), **different** (both sides have values, none match), **missing** (no value on at least one side). Country and nationality count together.

In [ ]:
_ID = re.compile(r"[^a-z0-9]")
def status(r, prop_list, kind="plain"):
    norm = (lambda v: _ID.sub("", str(v).lower())) if kind == "id" else (lambda v: str(v).strip().lower())
    L = {norm(v) for v in values(r, "left", prop_list)} - {""}
    R = {norm(v) for v in values(r, "right", prop_list)} - {""}
    if kind == "id":
        L, R = {x for x in L if len(x) >= 4}, {x for x in R if len(x) >= 4}
    if not L or not R:
        return "missing"
    if L & R or (kind == "dob" and any(a.startswith(b) or b.startswith(a) for a in L for b in R)):
        return "same"
    return "different"

t0 = time.time()
u = info.drop_duplicates("pair_id").set_index("pair_id")
u = u[u.index.isin(scores["pair_id"])]
rec = pd.DataFrame({"dob": [status(r, ["birthDate"], "dob") for r in u.itertuples(index=False)],
                    "country": [status(r, ["country", "nationality"]) for r in u.itertuples(index=False)],
                    "idnum": [status(r, ["idNumber"], "id") for r in u.itertuples(index=False)]}, index=u.index)
df = df.merge(rec, left_on="pair_id", right_index=True, how="left")
print(f"{len(rec):,} pairs in {time.time() - t0:.0f}s")
pd.concat({c: df.drop_duplicates("pair_id")[c].value_counts(normalize=True).round(3) for c in ["dob", "country", "idnum"]}, axis=1)

**Conclusion.**
- **ID numbers are almost always missing** (93% of pairs).
- **Birth date and country are present on both sides in about half of the pairs** (birth date 54%, country 60%). Where both sides have one, they usually agree: different in only 7% of all pairs.

**Next step:** none; this limits what any record feature can fix.

**What the table shows:** share of the validation pairs (each pair once) with same / different / missing birth date, country and ID number.

**How to read it:** "missing" on at least one side is the normal case for most fields; that limits how much any record feature can fix.

## 2 · Threshold and errors per split

For each of the 5 splits: the threshold that reaches 99% recall on that split's validation pairs, then
- **false alarm** = negative pair with score ≥ threshold,
- **miss** = positive pair with score < threshold.

In [ ]:
def threshold_at(y, s, recall):
    y, s = np.asarray(y), np.asarray(s)
    order = np.argsort(-s, kind="stable")
    k = min(int(np.searchsorted(np.cumsum(y[order]), np.ceil(recall * y.sum()))), len(y) - 1)
    return s[order][k]

thr = df.groupby("rep").apply(lambda g: threshold_at(g["y"], g["score"], TARGET), include_groups=False)
df["thr99"] = df["rep"].map(thr)
df["alarm"] = df["score"] >= df["thr99"]
df["fa"] = df["alarm"] & (df["y"] == 0)
df["miss"] = ~df["alarm"] & (df["y"] == 1)
t = df.groupby("rep").agg(pairs=("y", "size"), negatives=("y", lambda y: int((y == 0).sum())),
                          false_alarms=("fa", "sum"), misses=("miss", "sum"))
t["threshold"] = thr.round(4)
t["FA rate %"] = (100 * t["false_alarms"] / t["negatives"]).round(2)
t

**Conclusion.**
- **The false-alarm rates match notebook 07** (split by cluster, 2.98–3.44%, mean 3.29%), so the saved scores are the right ones.
- **Per split:** about 460–530 false alarms and 820–840 misses at 99% recall.

**Next step:** none.

**What the table shows:** per split, the number of validation pairs and negatives, the threshold for 99% recall, and the resulting false alarms and misses.

**How to read it:** the FA rate should match notebook 07 (section 10/11, split by cluster). The misses are about 1% of the positives by construction.

## 3 · Taxonomy of the false alarms

Every false alarm gets exactly **one** type, the first rule that matches (abbreviations of the features: notebook 04):

| Type | Rule |
| --- | --- |
| A1 identical name, separable | identical name or alias (`n_exact_name` = 1) **and** birth date, ID number or country **different**: the record fields could have told them apart |
| A2 identical name, not separable | identical name, and no record field different (missing or same country only): nothing in the data separates them |
| A3 identical name, label doubt | identical name **and** same birth date or same ID number: maybe the same entity after all (label error) |
| B company: same core, other legal form | company, core name similarity ≥ 0.95, legal forms conflict or only on one side |
| C different script | the two main names are in different scripts |
| D near-identical spelling | `n_sim_sort` ≥ 0.9 |
| E partial name | all words of the shorter name found in the longer one |
| F common name | rarest shared full name in 6+ entities or rarest shared word in more than 100 |
| G other | none of the above |

A3 is checked before A1 and A2 (a same birth date outweighs a different country).

In [ ]:
def fa_type(d):
    ident = d["n_exact_name"] == 1
    same_rec = (d["dob"] == "same") | (d["idnum"] == "same")
    diff_rec = (d["dob"] == "different") | (d["idnum"] == "different") | (d["country"] == "different")
    rules = [("A3 identical name, label doubt", ident & same_rec),
             ("A1 identical name, separable", ident & diff_rec),
             ("A2 identical name, not separable", ident),
             ("B company: same core, other legal form", ~d["person"] & (d["n_sim_core"] >= 0.95)
                                                        & d["n_legal_form"].isin(["conflict", "one_side"])),
             ("C different script", d["n_script"] == "different"),
             ("D near-identical spelling", d["n_sim_sort"] >= 0.9),
             ("E partial name", d["n_token_coverage"] == 1),
             ("F common name", (d["n_freq_name_band"] >= 2) | (d["n_freq_word_band"] >= 3))]
    out = pd.Series("G other", index=d.index)
    for name, rule in reversed(rules):
        out[rule.fillna(False).to_numpy()] = name
    return out

FA_TYPES = ["A1 identical name, separable", "A2 identical name, not separable", "A3 identical name, label doubt",
            "B company: same core, other legal form", "C different script", "D near-identical spelling",
            "E partial name", "F common name", "G other"]
neg = df[df["y"] == 0].copy()
neg["type"] = pd.Categorical(fa_type(neg), categories=FA_TYPES)
fa = neg[neg["fa"]]
per_rep = fa.groupby(["rep", "type"], observed=False).size().unstack("type")
tab = pd.DataFrame({"false alarms per split (mean)": per_rep.mean(), "sd": per_rep.std(),
                    "share": per_rep.sum() / per_rep.sum().sum()})
by = fa.groupby("type", observed=False).agg(persons=("person", "mean"), in_giant=("group", lambda g: (g == "giant").mean()))
tab.join(by).round(3)

**Conclusion.**
- **Identical names cause 72% of the false alarms** (A1–A3 together).
- **The largest single type is A2:** an identical name with no record field that differs. That is 270 per split, 55% of all false alarms, and no feature in this data can separate them.
- **A1 (14%) are pairs whose birth date, ID or country differ.** The model uses no record features yet (feature set "name"), so these are the false alarms the record features can remove.
- **A3 (2.4%) may be label errors.**
- **Among the name problems, partial names (E, 10%) and different scripts (C, 8%) lead.** Both sit mostly in the giant component (87%).

**Next step:** re-run with name + record features. A1 should shrink, A2 should stay.

**What the table shows:** false alarms at 99% recall per type: mean number per split and standard deviation over the 5 splits, share of all false alarms, share of persons and share inside the giant component.

**How to read it:** A1 is what a better use of the record fields could still fix; A2 is the floor no feature in this data can fix; A3 may not be errors at all. B–G are name problems.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
m = per_rep.mean()[::-1]; s = per_rep.std()[::-1]
m.index = s.index = [str(x) for x in m.index]
pers = fa.groupby("type", observed=False)["person"].mean()
pers.index = [str(x) for x in pers.index]
pers = pers.reindex(m.index).fillna(0)
ax.barh(m.index, m * pers, xerr=None, label="persons")
ax.barh(m.index, m * (1 - pers), left=m * pers, label="companies / organisations")
ax.errorbar(m, m.index, xerr=s, fmt="none", ecolor="black", capsize=3, lw=1)
ax.set_xlabel("false alarms per split at 99% recall (mean, bar = ±1 sd)")
ax.set_title("False alarms by type"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

**Conclusion.**
- **A2 dominates the chart,** mainly persons.
- **Companies matter for the name problems:** they make up the larger part of different script, near-identical spelling and partial name.
- **The bars are stable over the splits** (small error bars relative to the counts).

**Next step:** none.

**What the chart shows:** the same counts as the table, split into persons and companies, with the spread over the 5 splits.

## 4 · Taxonomy of the missed matches

Every miss (true match below the 99% threshold) gets one type, first rule that matches:

| Type | Rule |
| --- | --- |
| M1 conflicting birth date | birth dates on both sides, none match |
| M2 conflicting ID / country | ID numbers or countries on both sides, none match |
| M3 different script | the two main names are in different scripts |
| M4 company: core name differs | company with core name similarity < 0.9 |
| M5 partial name | shorter name only partly found in the longer one (word coverage below 1) |
| M6 very different names | `n_sim_sort` < 0.7 |
| M7 other | none of the above |

In [ ]:
def miss_type(d):
    rules = [("M1 conflicting birth date", d["dob"] == "different"),
             ("M2 conflicting ID / country", (d["idnum"] == "different") | (d["country"] == "different")),
             ("M3 different script", d["n_script"] == "different"),
             ("M4 company: core name differs", ~d["person"] & (d["n_sim_core"] < 0.9)),
             ("M5 partial name", d["n_token_coverage"] < 1),
             ("M6 very different names", d["n_sim_sort"] < 0.7)]
    out = pd.Series("M7 other", index=d.index)
    for name, rule in reversed(rules):
        out[rule.fillna(False).to_numpy()] = name
    return out

MISS_TYPES = ["M1 conflicting birth date", "M2 conflicting ID / country", "M3 different script",
              "M4 company: core name differs", "M5 partial name", "M6 very different names", "M7 other"]
pos = df[df["y"] == 1].copy()
pos["type"] = pd.Categorical(miss_type(pos), categories=MISS_TYPES)
ms = pos[pos["miss"]]
per_rep_m = ms.groupby(["rep", "type"], observed=False).size().unstack("type")
tab_m = pd.DataFrame({"misses per split (mean)": per_rep_m.mean(), "sd": per_rep_m.std(),
                      "share": per_rep_m.sum() / per_rep_m.sum().sum()})
tab_m.join(ms.groupby("type", observed=False).agg(persons=("person", "mean"),
                                                 in_giant=("group", lambda g: (g == "giant").mean()))).round(3)

**Conclusion.** The misses look very different from the false alarms:
- **Company names that differ in their core (M4, 35%) and different scripts (M3, 24%) dominate.**
- **"Other" (M7, 26%) is almost only persons,** mostly outside the giant component.
- **Conflicting birth dates or IDs (M1, M2) explain only 3% of the misses.**

**Next step:** read the examples (section 9). Several misses have completely different names; such positives were probably judged on other fields (address, registration), or are label errors.

**What the table shows:** missed matches at 99% recall per type, mean per split, standard deviation, share of all misses, share of persons and of the giant component.

**How to read it:** these pairs set the threshold. M1/M2 are true matches whose record fields contradict each other (wrong data in one list, or label errors); M3–M6 are name problems a better name comparison could fix.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
mm = per_rep_m.mean()[::-1]; sm = per_rep_m.std()[::-1]
mm.index = sm.index = [str(x) for x in mm.index]
ax.barh(mm.index, mm, xerr=sm, capsize=3, color="tab:red", alpha=0.8)
ax.set_xlabel("missed matches per split at 99% recall (mean ± sd)"); ax.set_title("Missed matches by type")
plt.tight_layout(); plt.show()

**Conclusion.** Company core names (M4), "other" persons (M7) and different scripts (M3) make up 85% of the misses; birth dates and very different names are minor.

**Next step:** better cross-script name comparison (transliteration) would address M3; M4 and M7 need a closer look at the examples.

## 5 · Error types along the cost curve

For recall from 90% to 99.9%, the FA rate split into the false-alarm types (mean over the 5 splits). This links the two deliverables: which kind of pair makes the last tenths of a percent of recall so expensive.

In [ ]:
grid = np.round(np.arange(0.90, 0.9991, 0.001), 4)
rows = []
for rep, g in df.groupby("rep"):
    gn = neg[neg["rep"] == rep]
    n_neg = len(gn)
    for r in grid:
        th = threshold_at(g["y"], g["score"], r)
        cnt = gn.loc[gn["score"] >= th, "type"].value_counts()
        rows.append({"rep": rep, "recall": r, **{t: 100 * cnt.get(t, 0) / n_neg for t in FA_TYPES}})
curve = pd.DataFrame(rows).groupby("recall")[FA_TYPES].mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.stackplot(curve.index * 100, [curve[t] for t in FA_TYPES], labels=FA_TYPES, alpha=0.9)
ax.axvline(99, color="grey", ls="--", lw=1); ax.axvline(99.5, color="grey", ls=":", lw=1)
ax.set_xlabel("recall (% of true matches found)"); ax.set_ylabel("false alarms (% of negative pairs)")
ax.set_title("Cost curve by error type"); ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

**Conclusion.**
- **The identical-name floor is flat:** A2 is already about 1.5% at 90% recall and grows only slowly. It does not cause the steep rise, it is the base under the whole curve.
- **The explosion beyond 99.5% recall comes from name problems:** different script (C), partial name (E) and "other" (G). To find the last true matches, the threshold drops so low that many name-wise dissimilar negatives get an alarm.

**Next step:** report the curve in this form; it shows the two different causes of cost (floor vs. tail).

**What the chart shows:** the cost curve (FA rate against recall) as stacked areas, one per false-alarm type. Dashed line: 99% recall, dotted: 99.5%.

**How to read it:** the type whose band widens most on the right is what forces the steep rise near 100% recall.

In [ ]:
pts = curve.loc[[0.95, 0.98, 0.99, 0.995, 0.999]]
pts.insert(0, "all", pts.sum(axis=1))
pts.index = [f"{r:.1%}" for r in pts.index]
pts.round(2)

**Conclusion.**

| Recall | false alarms | of which A2 (cannot be separated) |
| --- | --- | --- |
| 99% | 3.29% | 1.81 pp |
| 99.5% | 5.26% | 1.96 pp |
| 99.9% | 24.8% | 2.1 pp |

- **At 99%, more than half of the false alarms are A2:** a floor no name feature can lower.
- **At 99.9%, A2 is a small part;** the tail of name problems dominates.

**Next step:** state the floor in the report: at 99% recall, at least about 1.8% of the negatives get an alarm with these data.

**What the table shows:** the same numbers at five recall levels: total FA rate in % and its part per type.

**How to read it:** A2 (identical name, not separable) is the floor for this data: even a perfect use of every feature could not remove it at that recall level. Total minus A2 (and minus A3, if those are label errors) is what remains to be won.

## 6 · How far apart are the classes?

Score distribution of positive and negative validation pairs (split 0), logarithmic count axis, with the thresholds for 99% and 99.5% recall.

In [ ]:
g0 = df[df["rep"] == 0]
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 51)
ax.hist(g0.loc[g0["y"] == 1, "score"], bins=bins, alpha=0.6, label="positive (same entity)")
ax.hist(g0.loc[g0["y"] == 0, "score"], bins=bins, alpha=0.6, label="negative (different)")
for r, ls in [(0.99, "--"), (0.995, ":")]:
    ax.axvline(threshold_at(g0["y"], g0["score"], r), color="black", ls=ls, lw=1, label=f"threshold for {r:.1%} recall")
ax.set_yscale("log"); ax.set_xlabel("model score"); ax.set_ylabel("pairs (log scale)")
ax.set_title("Score distribution, split 0"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Conclusion.**
- **Most negatives get very low scores** and most positives very high ones.
- **But both classes have a thin tail into the other side:** positives with low scores (the misses) and negatives at the very top (identical names). The 99% threshold sits around 0.5, the 99.5% threshold clearly lower.

**Next step:** none.

**What the chart shows:** how many positive and negative pairs fall into each score band; the vertical lines are the thresholds.

**How to read it:** negatives to the right of a threshold are its false alarms. A thin tail of positives at low scores (the misses) pulls the threshold far to the left.

## 7 · Identical-name false alarms by record fields

The identical-name false alarms (types A1–A3) by birth-date status (rows) and country status (columns), summed over the 5 splits.

In [ ]:
a = fa[fa["n_exact_name"] == 1]
ct = pd.crosstab(a["dob"], a["country"], margins=True, margins_name="total")
fig, ax = plt.subplots(figsize=(5.5, 3.5))
im = ax.imshow(ct.iloc[:-1, :-1], cmap="Blues")
ax.set_xticks(range(ct.shape[1] - 1), ct.columns[:-1]); ax.set_yticks(range(ct.shape[0] - 1), ct.index[:-1])
ax.set_xlabel("country"); ax.set_ylabel("birth date")
for i in range(ct.shape[0] - 1):
    for j in range(ct.shape[1] - 1):
        ax.text(j, i, int(ct.iloc[i, j]), ha="center", va="center", fontsize=9)
ax.set_title("Identical-name false alarms"); plt.tight_layout(); plt.show()
ct

**Conclusion.**
- **Missing birth date and missing country (956) is the largest cell** of the identical-name false alarms, followed by missing birth date with the same country (459). Together 80%.
- **Only 221 have a different birth date** that a record feature could use, and 110 a different country.

**Next step:** with record features, the "different" row and column should mostly disappear; the "missing" cells stay.

**What it shows:** the number of identical-name false alarms per combination of birth-date and country status.

**How to read it:** the "missing / missing" cell is the hard floor. Cells with "different" are pairs the record fields could separate; if they are large while record features are in the model, the record features are not used well enough.

## 8 · Which features drive the false alarms? (SHAP)

The model of split 0 is trained again (same features, same setting as in notebook 07), and LightGBM's built-in SHAP values (`pred_contrib`) are computed for its validation pairs. Compared: **false alarms** against **correctly rejected negatives** (negatives below the threshold).

In [ ]:
sides = pd.read_parquet(C.DATA_PROCESSED / "splits_dev.parquet")
s0 = sides[sides["rep"] == 0].set_index("pair_id")["side"]
FCOLS = [c for c in feats.columns if c.startswith(("n_", "r_"))]
X = info.drop_duplicates("pair_id").set_index("pair_id")
X["is_person"] = X["person"].astype(int)
XCOLS = FCOLS + ["is_person"]
tr_ids, va_ids = s0.index[s0 == "train"], s0.index[s0 == "test"]
y_all = scores.drop_duplicates("pair_id").set_index("pair_id")["y"]
y_tr = pd.read_parquet(C.PAIRS_INPUT, columns=["pair_id", "judgement"]).set_index("pair_id")["judgement"].eq("positive").astype(int)
model = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                       random_state=C.RANDOM_SEED, verbose=-1).fit(X.loc[tr_ids, XCOLS], y_tr.loc[tr_ids])
Xv = X.loc[va_ids, XCOLS]
contrib = pd.DataFrame(model.predict(Xv, pred_contrib=True)[:, :-1], index=va_ids, columns=XCOLS)
sv = g0.set_index("pair_id")
same = np.allclose(model.predict_proba(Xv)[:, 1], sv.loc[va_ids, "score"], atol=1e-6)
print(f"retrained model reproduces the saved scores: {same}")
fa_ids = sv.index[sv["fa"]]; tn_ids = sv.index[(sv["y"] == 0) & ~sv["alarm"]]
shap_tab = pd.DataFrame({"mean SHAP, false alarms": contrib.loc[fa_ids].mean(),
                         "mean SHAP, correct negatives": contrib.loc[tn_ids].mean()})
shap_tab["difference"] = shap_tab.iloc[:, 0] - shap_tab.iloc[:, 1]
shap_tab = shap_tab.sort_values("difference", ascending=False)
fig, ax = plt.subplots(figsize=(7, 5))
shap_tab["difference"][::-1].plot.barh(ax=ax)
ax.set_xlabel("mean SHAP (log-odds): false alarms minus correct negatives")
ax.set_title("What pushes false alarms up (split 0)"); plt.tight_layout(); plt.show()
shap_tab.round(3)

**Conclusion.**
- **The retrained model reproduces the saved scores exactly,** so the SHAP values describe the same model.
- **False alarms look like matches above all through `n_sim_sort` and `n_token_coverage`:** the names are (almost) the same.
- **The full-name frequency (`n_freq_name_band`) is the only feature that pushes them down more than the correct negatives:** a common name is the remaining evidence against a match, but it is too weak. `n_exact_name` contributes nothing on its own (the other similarity scores carry it).

**Next step:** compare again with record features in; birth date should then carry most of the counter-evidence.

**What it shows:** for each feature, its average contribution (in log-odds) to the score of false alarms and of correctly rejected negatives, and the difference.

**How to read it:** features with a large positive difference are what make the false alarms look like matches (e.g. an identical name); features with a negative difference are the evidence that still argues against a match but was not strong enough.

## 9 · Examples per type

Up to 3 false alarms per type (highest scores first) and up to 3 misses per type (lowest scores first), split 0, with the main names of both sides.

In [ ]:
first = lambda v: list(v)[0] if v is not None and len(v) else ""
def examples(d, asc):
    e = d[d["rep"] == 0].sort_values("score", ascending=asc).groupby("type", observed=True).head(3)
    return e.assign(left=e["left_name"].map(first), right=e["right_name"].map(first))[
        ["type", "left", "right", "score", "dob", "country"]].sort_values("type").reset_index(drop=True)
display(examples(fa, asc=False))
display(examples(ms, asc=True))

**Conclusion.** The examples support the types and show two surprises:
- **Several "misses" have completely different names but are labelled positive** ("Leo Voss" vs "George Pylar"; nursing homes or companies with unrelated names). These were probably judged on other fields, such as the same address or registration in US exclusion lists, or they are label errors. No name feature can find them.
- **Some false alarms in "different script" are the same company name in Cyrillic and Latin labelled negative,** possibly label noise as well.

**Next step:** in the report, separate "name-findable" misses from misses only another field could find. Address and registration features (notebook 05) would address part of them.

**What it shows:** real pairs behind each type: false alarms (first table) and missed matches (second table), with score and record-field status.

**How to read it:** if the examples of a type look alike, the rule is a good category; if not, it should be split or reworded.

## Findings (uncleaned data, name features only, split by person cluster, 2026-11-25)

**False alarms at 99% recall: 3.29% of the negatives.**
- **72% are identical names.** More than half of all false alarms (55%, type A2) have no record field that differs: with these data no model can separate them. They form a flat floor of about 1.8% of the negatives at 99% recall.
- **14% are identical names with a different birth date, ID or country** (A1): the target of the record features.
- **The steep rise of the cost curve beyond 99.5% recall comes from name problems,** not from the floor: different scripts, partial names, other.

**Misses (the true matches that set the threshold):**
- mainly companies whose core names differ (35%), different scripts (24%) and other persons (26%);
- conflicting record fields explain only 3%;
- several positives have unrelated names: they were judged on other fields, or are label errors.

**What the model relies on (SHAP):** false alarms look like matches through near-identical names. The only counter-evidence in the name features is name frequency, which is too weak.

**Next steps:**
1. Re-run notebook 07 and this notebook on the cleaned data with name + record features. Expected: A1 shrinks, A2 stays as the floor.
2. Transliteration for cross-script pairs (types C and M3).
3. In the report, present the error taxonomy as **floor (A2) + fixable by record fields (A1) + name problems (B–G) + possible label errors (A3)**.
